# 02 – Configure

This notebook uses the **`azure-ai-inference` SDK** (new Azure AI Foundry SDK) to verify the
deployed model is ready before running test prompts.

For a simple model deployment like this one there is no index or data pipeline to set up.
In richer demos (e.g. RAG with AI Search) this notebook would create indexes, upload documents,
and register any additional resources.

**Pre-requisite:** `01_deploy_infra.ipynb` must have run and written `env/.env`.

---

## Research basis

> **`azure-ai-inference` vs classic `openai` SDK**
> The `azure-ai-inference` package (`ChatCompletionsClient`) is the Microsoft-native SDK for new
> Azure AI Foundry (AIServices) endpoints. It is model-agnostic — the same client works with
> OpenAI, Mistral, Cohere, and any other model in the Foundry catalog. The classic
> `openai.AzureOpenAI` SDK also works against AIServices endpoints via the `/openai` path,
> but is OpenAI-scoped and does not support multi-model routing.
>
> **Endpoint format**
> New Foundry AIServices: `https://<name>.services.ai.azure.com/`
> Classic Azure OpenAI:   `https://<name>.openai.azure.com/`
>
> **`[PREVIEW]` note** — `azure-ai-inference` is stable (GA), but the AIServices
> multi-model routing endpoint is considered *preview* for certain model families outside
> OpenAI. For OpenAI models (including `gpt-4.1-mini`) it is GA.

---

## Step 1 – Load environment variables

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

endpoint        = os.environ["AZURE_AI_ENDPOINT"]
api_key         = os.environ["AZURE_AI_API_KEY"]
deployment      = os.environ.get("AZURE_AI_DEPLOYMENT", "chat")
account_name    = os.environ.get("AZURE_ACCOUNT_NAME", "")
resource_group  = os.environ.get("AZURE_RESOURCE_GROUP", "")

print(f"Endpoint       : {endpoint}")
print(f"Deployment     : {deployment}")
print(f"Account name   : {account_name}")
print(f"Resource group : {resource_group}")

## Step 2 – Create the Azure AI Inference client

Using `azure-ai-inference.ChatCompletionsClient` — the new Azure AI Foundry SDK client.

In [ ]:
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(api_key),
)
print("✅  ChatCompletionsClient created (azure-ai-inference SDK)")

## Step 3 – Verify the deployment via Azure CLI

List deployments on the account and confirm ours is present and `Succeeded`.

In [ ]:
import subprocess, json

if account_name and resource_group:
    result = subprocess.run(
        ["az", "cognitiveservices", "account", "deployment", "list",
         "--name", account_name,
         "--resource-group", resource_group,
         "--query", "[].{name:name, state:properties.provisioningState, model:properties.model.name}",
         "--output", "json"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        for d in json.loads(result.stdout):
            status = "✅" if d.get("state") == "Succeeded" else "⚠️"
            print(f"{status}  {d['name']} ({d['model']}) — {d['state']}")
    else:
        print(f"CLI query failed: {result.stderr.strip()}")
else:
    print("AZURE_ACCOUNT_NAME or AZURE_RESOURCE_GROUP not set — skipping CLI check")

## Step 4 – Verify the deployment via API ping

A minimal single-token call to confirm the model endpoint responds.

In [ ]:
from azure.ai.inference.models import SystemMessage, UserMessage

response = client.complete(
    model=deployment,
    messages=[UserMessage(content="ping")],
    max_tokens=1,
)
print(f"✅  Deployment '{deployment}' responded — model: {response.model}")

---

## ✅ Deployment verified

The model is reachable and ready. Move to **`03_test.ipynb`** to run prompts.